<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/jobsheet%20dataset%20AntiBurnOut/JS06-ANN/07_Similarity_Search_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS06-ANN: Similarity Search (Approximate Nearest Neighbors) - Dataset AntiBurnOut

**Identitas Mahasiswa:**
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H

---

### Deskripsi Praktikum:
Praktikum ini menerapkan teknik **Approximate Nearest Neighbors (ANN)** untuk pencarian kesamaan profil responden (*Similarity Search*). Jika seorang pekerja memasukkan profil kesehatan mental dan beban kerjanya, sistem ANN akan secara instan mencari profil pekerja terdekat di database untuk merekomendasikan solusi intervensi (*wellness recommendation*) yang paling efektif.

### Langkah 1: Memuat Data dan Ekstraksi Vector Embeddings Profil

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import time

dataset_candidates = [
    'mental_health_burnout_prediction_dataset.csv',
    '../mental_health_burnout_prediction_dataset.csv',
    'jobsheet dataset AntiBurnOut/mental_health_burnout_prediction_dataset.csv'
]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), 'mental_health_burnout_prediction_dataset.csv')
df = pd.read_csv(dataset_path)

features = ['Work_Hours_Per_Week', 'Sleep_Hours', 'Anxiety_Score', 
            'Depression_Score', 'Mood_Score', 'Emotional_Stability', 'Physical_Activity_Hours']

X = df[features].copy()
for col in features:
    X[col] = X[col].fillna(X[col].median())

scaler = StandardScaler()
X_vectors = scaler.fit_transform(X).astype('float32')

print(f"Database Vektor Profil Siap: {X_vectors.shape[0]:,} entri dengan dimensi {X_vectors.shape[1]}")

### Langkah 2: Membangun Index Pencarian Tetangga Terdekat (Nearest Neighbors Index)

In [ ]:
# Menggunakan Scikit-learn NearestNeighbors dengan algoritma KD-Tree / Ball-Tree untuk pencarian cepat
start_time = time.time()
nn_model = NearestNeighbors(n_neighbors=5, metric='euclidean', algorithm='auto')
nn_model.fit(X_vectors)
build_time = time.time() - start_time

print(f"✅ Index Vektor berhasil dibangun dalam {build_time:.4f} detik.")

### Langkah 3: Simulasi Query Pencarian Profil Mirip & Rekomendasi Solusi

In [ ]:
# Profil responden uji (Pekerja dengan jam kerja 68 jam, tidur 4.5 jam, kecemasan 75, depresi 80)
query_profile = np.array([[68.0, 4.5, 75.0, 80.0, 25.0, 20.0, 1.0]])
query_scaled = scaler.transform(query_profile)

start_query = time.time()
distances, indices = nn_model.kneighbors(query_scaled)
query_latency = (time.time() - start_query) * 1000

print(f"Waktu pencarian query: {query_latency:.2f} ms\n")
print("Top-5 Profil Pekerja Paling Mirip di Database:")
recommendations = []
for rank, idx in enumerate(indices[0], 1):
    row = df.iloc[idx]
    rec = row['AI_Wellness_Recommendation']
    risk = row['Burnout_Risk']
    score = row['Burnout_Score']
    recommendations.append(rec)
    print(f"{rank}. Index #{idx} | Risiko: {risk} (Skor: {score}) | Rekomendasi: {rec}")

# Rekomendasi konsensus terbanyak
from collections import Counter
best_rec = Counter(recommendations).most_common(1)[0][0]
print(f"\n💡 Rekomendasi Sistem Terpilih untuk Pengguna: **{best_rec}**")

### Kesimpulan JS06-ANN:
1. Sistem pencarian berbasis ANN mampu menemukan 5 pekerja dengan profil kesehatan mental paling identik dalam waktu kurang dari beberapa milidetik.
2. Pendekatan ini memungkinkan personalisasi rekomendasi intervensi mental health (*wellness recommendation*) yang berbasis pada data riil pekerja serupa.